In [3]:
pip install langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 3.8 MB/s eta 0:00:00


Loading documents

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/content/SamplePDF-40kb-Text-7pages.pdf')
docs = loader.load()
print(len(docs))

print(docs[0].page_content)
print(docs[1].metadata)

7
T
THE KEYS OF THE CHEST
he little valley of Letterglas is a very green and very lonesome place
almost
always, for snow seldom lies on it, and the
few people who come into it depart again even sooner and as tracelessly, so
that its much
grass spreads from month to month uncovered
and untrodden. It runs westward—that is,
towards the ocean—but never reaches the shore, because a great grassy
curtain intervenes,
curved round the end of it, and prolonged in
the lower hill-ranges that bound it to north and south. They are just swarded
embankments
of most simple construction, with scarcely a
fold to complicate the sweep of their smooth
green slopes, and the outline of their ridge
against the sky undulates as softly as young
corn in a drowsy breeze. Only at one point—about midway on the right
hand looking
westward—it is suddenly broken by a sharp
dip down and up again, making a gap like
{'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-03-12T09:42:33+00:00'

Splitting

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

12


Embeddings

In [9]:
pip install -qU langchain-huggingface

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[0.03005211427807808, -0.037380918860435486, 0.005284505896270275, 0.07445745170116425, -0.08116881549358368, 0.04213171824812889, 0.014562860131263733, 0.010794274508953094, -0.0137054193764925, -0.046502359211444855]


Vector stores

In [12]:
pip install -qU langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.1 MB/s eta 0:00:00


In [13]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [14]:
ids = vector_store.add_documents(documents=all_splits)

In [17]:
results = vector_store.similarity_search(
    """Fifty years ago or more, you would have been likely enough any fine
morning to catch"""
)

print(results[0])

page_content='was abandoned on
a misty morning in April, more than fifty
springs ago; but the track itself is now
merely a most faint difference of shade in the sward, which has crept back
again indefatigably, even where the austere road-metal had
been thrown clattering down.
A day before that misty morning, if anybody had climbed up and had
passed through' metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-03-12T09:42:33+00:00', 'source': '/content/SamplePDF-40kb-Text-7pages.pdf', 'total_pages': 7, 'page': 1, 'page_label': '2', 'start_index': 794}


In [19]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("""Fifty years ago or more, you would have been likely enough any fine
morning to catch""")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 1.368293285369873

page_content='was abandoned on
a misty morning in April, more than fifty
springs ago; but the track itself is now
merely a most faint difference of shade in the sward, which has crept back
again indefatigably, even where the austere road-metal had
been thrown clattering down.
A day before that misty morning, if anybody had climbed up and had
passed through' metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-03-12T09:42:33+00:00', 'source': '/content/SamplePDF-40kb-Text-7pages.pdf', 'total_pages': 7, 'page': 1, 'page_label': '2', 'start_index': 794}


Return documents based on similarity to an embedded query

In [21]:
embedding = embeddings.embed_query("""how many years ago you would have been likely enough any fine
morning to catch""")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='was abandoned on
a misty morning in April, more than fifty
springs ago; but the track itself is now
merely a most faint difference of shade in the sward, which has crept back
again indefatigably, even where the austere road-metal had
been thrown clattering down.
A day before that misty morning, if anybody had climbed up and had
passed through' metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-03-12T09:42:33+00:00', 'source': '/content/SamplePDF-40kb-Text-7pages.pdf', 'total_pages': 7, 'page': 1, 'page_label': '2', 'start_index': 794}


Retrivers

In [23]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        """Where did Eileen Fitzmaurice, six or seven years old,had
lived with her mother and aunt?""",
        "Where is House in Glendoula",
    ],
)

[[Document(id='d08fd2c5-b3b5-4443-a4ed-246579625622', metadata={'producer': 'Pdftools SDK', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-03-12T09:42:33+00:00', 'source': '/content/SamplePDF-40kb-Text-7pages.pdf', 'total_pages': 7, 'page': 6, 'page_label': '7', 'start_index': 0}, page_content='Fifty years ago or more, you would have been likely enough any fine\nmorning to catch\nthe chief maker of this particular path in the act. If the years were, say, ten\nmore, it\nwould have proved to be a very little little-girl, whose brown hair held both\nsunshine and\nshadow, and whose hazel-green eyes were\nsoftly lit, and who in those early days of\nhers always wore an ugly reddish checked\npelisse, and a broad-brimmed straw hat with\nvelvet rosettes to match. This was little\nEileen Fitzmaurice, six or seven years old, who, ever since she could\nrecollect anything,\nand maybe some twelve months longer, had\nlived with her mother and aunt at the Big\nHouse in Glendoula. As you woul